# Minimal Marmousi FWI

This notebook keeps the full workflow small: load a Marmousi velocity model, generate observed data by forward modeling, then invert a smooth starting model for a few iterations.

Before running it, make sure the Marmousi `.npy` files exist under `examples/models/marmousi/`. If they do not, run the model preparation scripts from the examples documentation first.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch


def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / 'examples').is_dir() and (path / 'src' / 'sweep').is_dir():
            return path
    raise RuntimeError('Could not find the geophyai repository root. Start Jupyter from the repo or an examples subdirectory.')


repo = find_repo_root()
sys.path.insert(0, str(repo / 'src'))

from sweep.equations import Acoustic
from sweep.propagator.options import EagerOptions
from sweep.propagator.torch import PropTorch
from sweep.signal import ricker

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Load a Small Marmousi Window

The full prepared Marmousi model may be larger than needed for a first notebook. We crop a shallow window so the example runs quickly while still using the Marmousi model.

In [ ]:
true_path = repo / 'examples' / 'models' / 'marmousi' / 'true.npy'
smooth_path = repo / 'examples' / 'models' / 'marmousi' / 'smooth.npy'

if not true_path.exists() or not smooth_path.exists():
    raise FileNotFoundError(
        'Missing Marmousi true.npy/smooth.npy. Prepare them with the scripts in examples/models/marmousi first.'
    )

true_full = np.load(true_path).astype(np.float32)
smooth_full = np.load(smooth_path).astype(np.float32)

nz = min(96, true_full.shape[0])
nx = min(160, true_full.shape[1])
z0 = 0
x0 = max(0, true_full.shape[1] // 2 - nx // 2)

vp_true_np = true_full[z0 : z0 + nz, x0 : x0 + nx]
vp_init_np = smooth_full[z0 : z0 + nz, x0 : x0 + nx]
shape = vp_true_np.shape
print('model shape:', shape)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
for ax, model, title in zip(axes, [vp_true_np, vp_init_np], ['true Marmousi', 'smooth start']):
    im = ax.imshow(model, cmap='seismic', aspect='auto')
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

## 2. Build Geometry, Wavelet, and Solver

In [ ]:
dh = 25.0
dt = 0.002
nt = 1000
freq = 6.0
delay = 0.18

nshots = 3
nreceivers = min(80, shape[1])

src_x = np.linspace(shape[1] * 0.2, shape[1] * 0.8, nshots).astype(np.int64)
sources = np.stack([src_x, np.full(nshots, 2, dtype=np.int64)], axis=1)

rec_x = np.linspace(0, shape[1] - 1, nreceivers).astype(np.int64)
receivers_one_shot = np.stack([rec_x, np.full(nreceivers, 4, dtype=np.int64)], axis=1)
receivers = np.repeat(receivers_one_shot[None, :, :], nshots, axis=0)

t = np.arange(nt, dtype=np.float32) * dt
wavelet = ricker(t - delay, f=freq).astype(np.float32)

equation = Acoustic(spatial_order=4, device=device, backend='torch')
solver = PropTorch(
    equation,
    shape=shape,
    dh=dh,
    dt=dt,
    dev=device,
    abcn=20,
    source_type=['h1'],
    receiver_type=['h1'],
    pml_type='cpmlr',
    backend='torch',
    impl='eager',
    eager_options=EagerOptions(use_compile=True),
    use_ckpt=False,
)

plt.figure(figsize=(6, 2.5))
plt.plot(t, wavelet)
plt.title('Ricker wavelet')
plt.xlabel('time (s)')
plt.tight_layout()
plt.show()

## 3. Forward Modeling: Create Observed Data

In [ ]:
vp_true = torch.tensor(vp_true_np, dtype=torch.float32, device=device)

with torch.no_grad():
    observed = solver(wavelet, sources, receivers, models=[vp_true]).detach()

print('observed shape:', tuple(observed.shape))

shot0 = observed[0].detach().cpu().numpy().squeeze()
gather0 = shot0.T if shot0.shape[-1] == nt else shot0

plt.figure(figsize=(7, 3.5))
clip = np.percentile(np.abs(gather0), 98)
plt.imshow(gather0, cmap='seismic', vmin=-clip, vmax=clip, aspect='auto')
plt.title('observed gather, shot 0')
plt.xlabel('receiver')
plt.ylabel('time sample')
plt.colorbar(shrink=0.8)
plt.tight_layout()
plt.show()

## 4. Invert the Smooth Model

This is deliberately tiny: all shots are used every iteration, and the loop is just Adam + mean-squared error.

In [ ]:
vp = torch.tensor(vp_init_np, dtype=torch.float32, device=device, requires_grad=True)
optimizer = torch.optim.Adam([vp], lr=25.0, eps=1e-16)
losses = []

for iteration in range(50):
    optimizer.zero_grad()
    predicted = solver(wavelet, sources, receivers, models=[vp])
    loss = (predicted - observed).pow(2).mean()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
    print(f'iteration {iteration:02d} | loss {losses[-1]:.6e}')

## 5. Look at the Result

In [ ]:
vp_final = vp.detach().cpu().numpy()
vmin = float(min(vp_true_np.min(), vp_init_np.min(), vp_final.min()))
vmax = float(max(vp_true_np.max(), vp_init_np.max(), vp_final.max()))
x_trace = shape[1] // 2
z_axis = np.arange(shape[0]) * dh

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), constrained_layout=True)
for ax, model, title in zip(
    axes[:3],
    [vp_true_np, vp_init_np, vp_final],
    ['true', 'initial', 'after 8 iterations'],
):
    im = ax.imshow(model, cmap='seismic', vmin=vmin, vmax=vmax, aspect='auto')
    ax.axvline(x_trace, color='yellow', linewidth=1.2, linestyle='--')
    ax.set_title(title)
    ax.set_xlabel('x sample')
    ax.set_ylabel('z sample')
    fig.colorbar(im, ax=ax, shrink=0.75)

axes[3].plot(losses, marker='o')
axes[3].set_title('loss')
axes[3].set_xlabel('iteration')
axes[3].set_yscale('log')
axes[3].grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(5, 4))
plt.plot(vp_true_np[:, x_trace], z_axis, label='true')
plt.plot(vp_init_np[:, x_trace], z_axis, label='initial')
plt.plot(vp_final[:, x_trace], z_axis, label='inverted')
plt.gca().invert_yaxis()
plt.title(f'model trace at x={x_trace}')
plt.xlabel('velocity (m/s)')
plt.ylabel('depth (m)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()